In [1]:
# Run this cell first: add local mmdet to path and register mmdet scope.
# If you get "RuntimeError: cannot cache function ... no locator available" (imgaug/numba),
# run the kernel with env: NUMBA_DISABLE_JIT=1 or skip this cell and use device='cpu' in the next cell.
import sys
import os
_mmdet_root = os.path.join(os.getcwd(), 'onedl-mmdetection')
if _mmdet_root not in sys.path:
    sys.path.insert(0, _mmdet_root)

# Import mmdet so its registries (scope "mmdet") exist in mmengine
import mmdet.registry  # noqa: F401
from mmdeploy.codebase import import_codebase
from mmdeploy.utils import Codebase
import_codebase(Codebase.MMDET)

/home/serene/.local/lib/python3.11/site-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


Could not import OneDL
Could not import OneDL


In [2]:
import mmengine
from mmdeploy.apis import inference_model

# Load model config and set default_scope='mmdet' (base config has default_scope=None, which causes "scope None" warnings)
model_cfg = mmengine.Config.fromfile('/workspaces/mowing-terrain-seg/work_dirs/ycor-lm-3cls-exps-0/mask2former_r50_8xb2-90k_ycor-1024x544.py')
model_cfg.default_scope = 'mmseg'

result = inference_model(
  model_cfg=model_cfg,
  deploy_cfg='/workspaces/mowing-terrain-seg/configs/deploy/custom/segmentation_onnxruntime_dynamic.py',
  backend_files=['/workspaces/mowing-terrain-seg/mmdeploy_model/mask2former-onnx/end2end.onnx'],
  img='onedl-mmdetection/demo/demo.jpg',
  device='cuda:0')

02/09 07:31:12 - mmengine - WARNING - Failed to search registry with scope "mmdet" in the "Codebases" registry tree. As a workaround, the current "Codebases" registry in "mmdeploy" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmdet" is a correct scope, or whether the registry is initialized.
Could not import OneDL
Could not import OneDL
02/09 07:31:12 - mmengine - WARNING - Failed to search registry with scope "mmseg" in the "Codebases" registry tree. As a workaround, the current "Codebases" registry in "mmdeploy" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmseg" is a correct scope, or whether the registry is initialized.
02/09 07:31:12 - mmengine - WARNING - Failed to search registry with scope "mmseg" in the "mmseg_tasks" registry tree. As a workaround, the current "mmseg_tasks" registry in "mmdeploy" is used to build instance. This may cause u

2026-02-09 07:31:12.922612065 [W:onnxruntime:, transformer_memcpy.cc:74 ApplyImpl] 68 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2026-02-09 07:31:12.933154917 [W:onnxruntime:, session_state.cc:1166 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2026-02-09 07:31:12.933164660 [W:onnxruntime:, session_state.cc:1168 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.


In [4]:
import onnx

model_path = "/workspaces/mowing-terrain-seg/mmdeploy_model/mask2former-onnx/end2end.onnx"
model = onnx.load(model_path)

# All op types and counts
from collections import Counter
op_types = Counter(node.op_type for node in model.graph.node)
for op, count in sorted(op_types.items(), key=lambda x: -x[1]):
    print(f"{count:5d}  {op}")

# Nodes that are not standard ONNX (likely CPU or custom)
custom_or_uncommon = [n for n in model.graph.node if "::" in n.op_type or n.op_type not in (
    "Conv", "Relu", "Add", "Mul", "MatMul", "BatchNormalization", "Clip", "Resize", "GridSample", ...)]
print("\n--- Nodes likely custom / CPU ---")
for n in custom_or_uncommon[:50]:  # first 50
    print(n.op_type, n.name)

 1878  Constant
  610  Unsqueeze
  397  Add
  320  Shape
  306  Concat
  260  Reshape
  246  Gather
  190  Transpose
  183  Mul
  175  MatMul
  174  Slice
  121  Cast
  100  Div
   98  ReduceMean
   85  Relu
   61  Sub
   59  Conv
   49  Pow
   49  Sqrt
   27  Identity
   27  ConstantOfShape
   25  Softmax
   18  Expand
   18  grid_sampler
   18  Gemm
   15  Where
   15  ReduceSum
   12  CumSum
   12  Sin
   12  Cos
   12  Tile
   11  Resize
   11  Einsum
   10  Sigmoid
    9  Squeeze
    9  Less
    9  Equal
    9  Not
    9  And
    6  Range
    5  InstanceNormalization
    4  Split
    3  Flatten
    1  MaxPool
    1  ArgMax

--- Nodes likely custom / CPU ---
Identity Identity_1878
Identity Identity_1879
Identity Identity_1880
Identity Identity_1881
Identity Identity_1882
Identity Identity_1883
Identity Identity_1884
Identity Identity_1885
Identity Identity_1886
Identity Identity_1887
Identity Identity_1888
Identity Identity_1889
Identity Identity_1890
Identity Identity_1891
Identit

In [5]:
grid_sampler_nodes = [n for n in model.graph.node if n.op_type == 'grid_sampler']
print("Count:", len(grid_sampler_nodes))
if grid_sampler_nodes:
    n = grid_sampler_nodes[0]
    print("domain:", n.domain)   # likely "mmdeploy"
    print("op_type:", n.op_type) # "grid_sampler"

Count: 18
domain: mmdeploy
op_type: grid_sampler
